# Embedding Model Smoke Test

This notebook compares candidate embedding models for:

1. Patient-case retrieval
2. Knowledge-base retrieval

Candidate models:
- BAAI/bge-base-en-v1.5
- ncbi/MedCPT
- nomic-ai/nomic-embed-text-v1.5

Retrieval baseline:
- BM25

In [ ]:
import numpy as np
from typing import List

In [ ]:
# def embed_with_sentence_transformer(model_name: str, texts: List[str]) -> np.ndarray:
#     from sentence_transformers import SentenceTransformer

#     model = SentenceTransformer(model_name)

#     embeddings = model.encode(
#         texts,
#         normalize_embeddings=True,
#         show_progress_bar=False
#     )

#     return np.array(embeddings)


# def embed_with_medcpt(texts: List[str], is_query: bool) -> np.ndarray:
#     from transformers import AutoTokenizer, AutoModel
#     import torch

#     model_name = (
#         "ncbi/MedCPT-Query-Encoder"
#         if is_query
#         else "ncbi/MedCPT-Article-Encoder"
#     )

#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     model = AutoModel.from_pretrained(model_name)
#     model.eval()

#     with torch.no_grad():
#         encoded = tokenizer(
#             texts,
#             truncation=True,
#             padding=True,
#             return_tensors="pt",
#             max_length=512
#         )

#         embeds = model(**encoded).last_hidden_state[:, 0, :]

#     embeds = embeds.numpy()

#     embeds = embeds / np.linalg.norm(
#         embeds,
#         axis=1,
#         keepdims=True
#     )

#     return embeds


# MODEL_REGISTRY = {
#     "bge-base-en-v1.5": lambda texts, is_query: embed_with_sentence_transformer(
#         "BAAI/bge-base-en-v1.5", texts
#     ),

#     "nomic-embed-text": lambda texts, is_query: embed_with_sentence_transformer(
#         "nomic-ai/nomic-embed-text-v1.5", texts
#     ),

#     "medcpt": embed_with_medcpt,
# }


import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel


print("Loading BGE...")
bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("Loading Nomic...")
nomic_model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

print("Loading MedCPT query encoder...")
medcpt_query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)
medcpt_query_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Query-Encoder"
)
medcpt_query_model.eval()

print("Loading MedCPT article encoder...")
medcpt_article_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Article-Encoder"
)
medcpt_article_model = AutoModel.from_pretrained(
    "ncbi/MedCPT-Article-Encoder"
)
medcpt_article_model.eval()

print("All models loaded.")

In [ ]:
def embed_with_bge(texts, is_query=False, batch_size=32):

    embeddings = bge_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return np.array(embeddings)


def embed_with_nomic(texts, is_query=False, batch_size=32):

    prefix = "search_query: " if is_query else "search_document: "

    prefixed_texts = [
        prefix + text
        for text in texts
    ]

    embeddings = nomic_model.encode(
        prefixed_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [ ]:
from tqdm.auto import tqdm

def embed_with_medcpt(texts, is_query=False, batch_size=32):

    if is_query:
        tokenizer = medcpt_query_tokenizer
        model = medcpt_query_model
    else:
        tokenizer = medcpt_article_tokenizer
        model = medcpt_article_model

    all_embeddings = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="MedCPT batches"
    ):

        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        )

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = outputs.last_hidden_state[:, 0, :]

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

    return np.vstack(all_embeddings)

In [ ]:
MODEL_REGISTRY = {
    "bge-base-en-v1.5": embed_with_bge,
    "nomic-embed-text": embed_with_nomic,
    "medcpt": embed_with_medcpt,
}

EMBEDDING_MODELS = [
    "bge-base-en-v1.5",
    "nomic-embed-text",
    "medcpt"
]

# Part 1 — Patient-Case Retrieval Test

This test evaluates the ability of each candidate embedding model to retrieve
clinically relevant evidence from a patient's own case chunks.

Five admissions are selected for the smoke test. For each admission, all chunks
belonging to that admission form its retrieval corpus.

In [ ]:
import json
from collections import defaultdict

PATIENT_CHUNKS_PATH = "all_patient_chunks_v2.jsonl"

patient_chunks = []

with open(PATIENT_CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            patient_chunks.append(json.loads(line))

print(f"Total patient chunks loaded: {len(patient_chunks)}")

In [ ]:
print("Example chunk:")
print(patient_chunks[0])

print("\nAvailable fields:")
print(patient_chunks[0].keys())

In [ ]:
SELECTED_ADMISSIONS = [
    (00000000, 00000000),
    (00000000, 00000000),
    (00000000, 00000000),
    (00000000, 00000000),
    (00000000, 00000000),
]

# Group chunks belonging to the selected admissions
selected_patient_chunks = defaultdict(list)

for chunk in patient_chunks:
    admission_key = (
        chunk["subject_id"],
        chunk["hadm_id"]
    )

    if admission_key in SELECTED_ADMISSIONS:
        selected_patient_chunks[admission_key].append(chunk)

In [ ]:
for admission in SELECTED_ADMISSIONS:
    chunks = selected_patient_chunks[admission]

    print(
        f"subject_id={admission[0]}, "
        f"hadm_id={admission[1]} -> "
        f"{len(chunks)} chunks"
    )

In [ ]:
# Inspecting the chunks before creating queries:
for admission in SELECTED_ADMISSIONS:
    print("\n" + "=" * 100)
    print(f"SUBJECT_ID: {admission[0]} | HADM_ID: {admission[1]}")
    print("=" * 100)

    for chunk in selected_patient_chunks[admission]:
        print(
            f"\nChunk ID: {chunk['chunk_id']} | "
            f"Section: {chunk['section']} | "
            f"Type: {chunk['chunk_type']}"
        )
        print(chunk["text"])
        print("-" * 100)

In [ ]:
# Step 6 — Create a compact chunk index
def show_chunk_index(admission, preview_chars=180):
    chunks = selected_patient_chunks[admission]

    print(
        f"\nSUBJECT_ID: {admission[0]} | "
        f"HADM_ID: {admission[1]} | "
        f"Total chunks: {len(chunks)}"
    )
    print("=" * 100)

    for chunk in chunks:
        preview = chunk["text"].replace("\n", " ")[:preview_chars]

        print(
            f"ID={chunk['chunk_id']:>3} | "
            f"Section={chunk['section']:<30} | "
            f"{preview}"
        )

In [ ]:
show_chunk_index(SELECTED_ADMISSIONS[0])

In [ ]:
show_chunk_index(SELECTED_ADMISSIONS[1])

In [ ]:
show_chunk_index(SELECTED_ADMISSIONS[2])

In [ ]:
show_chunk_index(SELECTED_ADMISSIONS[3])

In [ ]:
show_chunk_index(SELECTED_ADMISSIONS[4])

In [ ]:
# Helper function to display the full text of an identified chunk, for identified patient

def show_chunk(admission, chunk_id):
    for chunk in selected_patient_chunks[admission]:
        if chunk["chunk_id"] == chunk_id:
            print(f"Section: {chunk['section']}")
            print(f"Chunk ID: {chunk['chunk_id']}")
            print("\n" + chunk["text"])
            return

    print("Chunk not found.")

In [ ]:
RETRIEVAL_TESTS = [
    {
    "admission": (00000000, 00000000),
    "query": "what are the abnormal laboratory results of this patient?",
    "relevant_chunk_ids": [15,16,17,18]
},
{
    "admission": (00000000, 00000000),
    "query": "what kidney problem did the patient present with?",
    "relevant_chunk_ids": [2,3]
},
{
    "admission": (00000000, 00000000),
    "query":"Was a renal transplant biopsy performed, and what was the procedural outcome?",
    "relevant_chunk_ids": [52]
},


  {
  "admission": (00000000,00000000),
  "query": "What symptoms the patient was found to have in both lung bases?", 
  "relevant_chunk_ids": [6]
},
{
  "admission": (00000000,00000000),
  "query": "What medications were administered to the patient during this admission?", 
  "relevant_chunk_ids": [30,31,32,33,34,35,36]   # new after medication cleaning are [18, 19, 20, 21, 22, 23, 24, 25]  (no need to correct them here as the code will handel it)
},
{
  "admission": (00000000,00000000),
  "query": "Why no contrast was given to the patient during the imaging?", 
  "relevant_chunk_ids": [143]
},


  {
    "admission": (00000000, 00000000),
    "query": "what is the gender of this patient",
    "relevant_chunk_ids": [1]
},
{
    "admission": (00000000, 00000000),
    "query": "what are the past medical conditions this patient have",
    "relevant_chunk_ids": [2]
},
{
    "admission": (00000000, 00000000),
    "query": "what does the ERCP note of this patient show",
    "relevant_chunk_ids": [51]
},


{
    "admission": (00000000, 00000000),
    "query": "Did this patient present with diarrhea?",
    "relevant_chunk_ids": [4]
},
{
    "admission": (00000000, 00000000),
    "query": "Were there any abnormal lung findings on physical examination?",
    "relevant_chunk_ids": [7]
},
{
    "admission": (00000000, 00000000),
    "query": "What medications did this patient have during admission",
    "relevant_chunk_ids": [12,13,14,15,16,17,18,19,20]    # new after medication cleaning are [12, 13, 14, 15, 16, 17, 18, 19] the same explanation as query#5
},


{
    "admission": (00000000, 00000000),
    "query": "what is the gender of this patient",
    "relevant_chunk_ids": [1]
},
{
    "admission": (00000000, 00000000),
    "query": "did this patient present any cough",
    "relevant_chunk_ids": [2]
},
{
    "admission": (00000000, 00000000),
    "query": "what are the procedures this patient had during this admission",
    "relevant_chunk_ids": [8]
}
]

In [ ]:
# ============================================================
# Map old patient labels to the v2 chunk_ids by exact text match.
# Chunks from unchanged sections map automatically; chunks from
# the cleaned sections (medications, input events) need manual relabelling.
# ============================================================
import json
from collections import defaultdict

def load_by_admission(path, admissions):
    out = defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            key = (c["subject_id"], c["hadm_id"])
            if key in admissions:
                out[key][c["chunk_id"]] = c
    return out

adm = set(SELECTED_ADMISSIONS)
old_by = load_by_admission("all_patient_chunks.jsonl", adm)
new_by = load_by_admission(PATIENT_CHUNKS_PATH, adm)

needs_manual = []
for i, test in enumerate(RETRIEVAL_TESTS, start=1):
    a = test["admission"]
    text_to_new = {c["text"]: cid for cid, c in new_by[a].items()}
    new_ids, missing = [], []
    for old_id in test["relevant_chunk_ids"]:
        old = old_by[a][old_id]
        nid = text_to_new.get(old["text"])
        if nid is not None:
            new_ids.append(nid)
        else:
            missing.append((old_id, old["section"]))
    test["relevant_chunk_ids_old"] = test["relevant_chunk_ids"]
    test["relevant_chunk_ids"] = new_ids
    status = "OK" if not missing else f"MANUAL - changed: {missing}"
    print(f"{i:>2} | {test['query'][:45]:45s} | {test['relevant_chunk_ids_old']} -> {new_ids} | {status}")
    if missing:
        needs_manual.append(i)

print("\nTests needing manual relabelling:", needs_manual)

In [ ]:
# ============================================================
# Relabel the medication queries: same rule as the original labels
# (the whole medication section is the answer), applied to the merged
# MEDICATIONS section in the v2 file.
# ============================================================
for t_idx in needs_manual:
    test = RETRIEVAL_TESTS[t_idx - 1]
    a = test["admission"]
    med_ids = sorted(cid for cid, c in new_by[a].items() if c["section"] == "MEDICATIONS")
    test["relevant_chunk_ids"] = med_ids
    first = new_by[a][med_ids[0]]["text"].split("\n")[0][:80] if med_ids else "(none)"
    print(f"{t_idx:>2} | {test['query'][:45]:45s} | new ids: {med_ids} | first line: {first}")

# Save the final v2 labels so the test is reproducible without re-running the mapping
import json
with open("smoke_test_labels_v2.json", "w", encoding="utf-8") as f:
    json.dump([{**t, "admission": list(t["admission"])} for t in RETRIEVAL_TESTS],
              f, indent=2, ensure_ascii=False)
print("\nSaved: smoke_test_labels_v2.json")

In [ ]:
# Add the evaluation metrics
def mrr_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    for rank, chunk_id in enumerate(ranked_ids[:k], start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def recall_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    retrieved_relevant = len(
        set(ranked_ids[:k]) & relevant_ids
    )

    return retrieved_relevant / len(relevant_ids)

In [ ]:
TOP_K = 10

In [ ]:
# Retrieve chunks with an embedding model
# Query → embed query → embed that patient's chunks → calculate similarity → return the 10 most relevant chunk IDs.

def retrieve_with_embedding(model_name, query, chunks, top_k=10):
    """
    Rank a patient's chunks by similarity to the query.
    """

    chunk_texts = [chunk["text"] for chunk in chunks]
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    embed_function = MODEL_REGISTRY[model_name]

    # Embed patient chunks
    chunk_embeddings = embed_function(
        chunk_texts,
        is_query=False
    )

    # Embed query
    query_embedding = embed_function(
        [query],
        is_query=True
    )[0]

    # Embeddings are normalized, so dot product = cosine similarity
    scores = chunk_embeddings @ query_embedding

    # Rank from highest to lowest similarity
    ranked_indices = np.argsort(scores)[::-1]

    ranked_ids = [
        chunk_ids[i]
        for i in ranked_indices[:top_k]
    ]

    return ranked_ids
    

In [ ]:
# Testing one query
test = RETRIEVAL_TESTS[0]

admission = test["admission"]
query = test["query"]
relevant_ids = test["relevant_chunk_ids"]

chunks = selected_patient_chunks[admission]

ranked_ids = retrieve_with_embedding(
    model_name="bge-base-en-v1.5",
    query=query,
    chunks=chunks,
    top_k=TOP_K
)

print("Query:")
print(query)

print("\nGround-truth relevant chunks:")
print(relevant_ids)

print("\nTop-10 retrieved chunks:")
print(ranked_ids)

print("\nRecall@1:", recall_at_k(ranked_ids, relevant_ids, 1))
print("Recall@3:", recall_at_k(ranked_ids, relevant_ids, 3))
print("Recall@5:", recall_at_k(ranked_ids, relevant_ids, 5))
print("Recall@10:", recall_at_k(ranked_ids, relevant_ids, 10))
print("MRR@10:", mrr_at_k(ranked_ids, relevant_ids, 10))

In [ ]:
# build BM25 retrieval
def retrieve_with_bm25(query, chunks, top_k=10):
    """
    Rank a patient's chunks using BM25 lexical retrieval.
    """
    
    from rank_bm25 import BM25Okapi

    chunk_texts = [chunk["text"] for chunk in chunks]
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    # Simple tokenization
    tokenized_corpus = [
        text.lower().split()
        for text in chunk_texts
    ]

    tokenized_query = query.lower().split()

    # Calculate BM25 scores
    bm25 = BM25Okapi(tokenized_corpus)
    scores = bm25.get_scores(tokenized_query)

    # Rank highest score first
    ranked_indices = np.argsort(scores)[::-1]

    ranked_ids = [
        chunk_ids[i]
        for i in ranked_indices[:top_k]
    ]

    return ranked_ids

In [ ]:
!pip install rank-bm25

In [ ]:
# Test the same query with BM25
bm25_ranked_ids = retrieve_with_bm25(
    query=query,
    chunks=chunks,
    top_k=TOP_K
)

print("Ground-truth relevant chunks:")
print(relevant_ids)

print("\nBM25 Top-10 retrieved chunks:")
print(bm25_ranked_ids)

print("\nRecall@1:", recall_at_k(bm25_ranked_ids, relevant_ids, 1))
print("Recall@3:", recall_at_k(bm25_ranked_ids, relevant_ids, 3))
print("Recall@5:", recall_at_k(bm25_ranked_ids, relevant_ids, 5))
print("Recall@10:", recall_at_k(bm25_ranked_ids, relevant_ids, 10))
print("MRR@10:", mrr_at_k(bm25_ranked_ids, relevant_ids, 10))


In [ ]:
# Test the same query with MedCPT

medcpt_ranked_ids = retrieve_with_embedding(
    model_name="medcpt",
    query=query,
    chunks=chunks,
    top_k=TOP_K
)

print("MedCPT:", medcpt_ranked_ids)
print("Recall@10:", recall_at_k(medcpt_ranked_ids, relevant_ids, 10))
print("MRR@10:", mrr_at_k(medcpt_ranked_ids, relevant_ids, 10))

In [ ]:
#  Test the same query with MedCPT

nomic_ranked_ids = retrieve_with_embedding(
    model_name="nomic-embed-text",
    query=query,
    chunks=chunks,
    top_k=TOP_K
)

print("Nomic:", nomic_ranked_ids)
print("Recall@10:", recall_at_k(nomic_ranked_ids, relevant_ids, 10))
print("MRR@10:", mrr_at_k(nomic_ranked_ids, relevant_ids, 10))

In [ ]:
MODEL_REGISTRY = {
    "bge-base-en-v1.5": embed_with_bge,
    "nomic-embed-text": embed_with_nomic,
    "medcpt": embed_with_medcpt,
}

EMBEDDING_MODELS = [
    "bge-base-en-v1.5",
    "nomic-embed-text",
    "medcpt"
]

In [ ]:
print(MODEL_REGISTRY)

In [ ]:
# Pre-compute patient-chunk embeddings
embedding_cache = {}

EMBEDDING_MODELS = [
    "bge-base-en-v1.5",
    "nomic-embed-text",
    "medcpt"
]

for model_name in EMBEDDING_MODELS:

    print(f"\nEmbedding chunks with {model_name}...")

    embed_function = MODEL_REGISTRY[model_name]

    for admission in SELECTED_ADMISSIONS:

        chunks = selected_patient_chunks[admission]
        chunk_texts = [chunk["text"] for chunk in chunks]
        chunk_ids = [chunk["chunk_id"] for chunk in chunks]

        embeddings = embed_function(
            chunk_texts,
            is_query=False
        )

        embedding_cache[(model_name, admission)] = {
            "chunk_ids": chunk_ids,
            "embeddings": embeddings
        }

    print(f"Finished {model_name}")

In [ ]:
# Retrieval using the cache
def retrieve_with_cached_embeddings(
    model_name,
    admission,
    query,
    top_k=10
):
    embed_function = MODEL_REGISTRY[model_name]

    cached = embedding_cache[(model_name, admission)]

    chunk_ids = cached["chunk_ids"]
    chunk_embeddings = cached["embeddings"]

    # Only the query needs to be embedded now
    query_embedding = embed_function(
        [query],
        is_query=True
    )[0]

    scores = chunk_embeddings @ query_embedding

    ranked_indices = np.argsort(scores)[::-1]

    ranked_ids = [
        chunk_ids[i]
        for i in ranked_indices[:top_k]
    ]

    return ranked_ids

In [ ]:
# Run the complete evaluation
assert all("relevant_chunk_ids_old" in t for t in RETRIEVAL_TESTS), "Run the mapping and relabel cells first"

results = []

for test_number, test in enumerate(RETRIEVAL_TESTS, start=1):

    admission = test["admission"]
    query = test["query"]
    relevant_ids = test["relevant_chunk_ids"]

    chunks = selected_patient_chunks[admission]

    # Dense embedding models
    for model_name in EMBEDDING_MODELS:

        ranked_ids = retrieve_with_cached_embeddings(
            model_name=model_name,
            admission=admission,
            query=query,
            top_k=TOP_K
        )

        results.append({
            "test": test_number,
            "model": model_name,
            "recall@1": recall_at_k(ranked_ids, relevant_ids, 1),
            "recall@3": recall_at_k(ranked_ids, relevant_ids, 3),
            "recall@5": recall_at_k(ranked_ids, relevant_ids, 5),
            "recall@10": recall_at_k(ranked_ids, relevant_ids, 10),
            "mrr@10": mrr_at_k(ranked_ids, relevant_ids, 10)
        })

    # BM25
    ranked_ids = retrieve_with_bm25(
        query=query,
        chunks=chunks,
        top_k=TOP_K
    )

    results.append({
        "test": test_number,
        "model": "BM25",
        "recall@1": recall_at_k(ranked_ids, relevant_ids, 1),
        "recall@3": recall_at_k(ranked_ids, relevant_ids, 3),
        "recall@5": recall_at_k(ranked_ids, relevant_ids, 5),
        "recall@10": recall_at_k(ranked_ids, relevant_ids, 10),
        "mrr@10": mrr_at_k(ranked_ids, relevant_ids, 10)
    })

print(f"Finished. Total evaluations: {len(results)}")

In [ ]:
# Summarize the results
import pandas as pd

results_df = pd.DataFrame(results)

summary_df = (
    results_df
    .groupby("model")[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr@10"]
    ]
    .mean()
    .round(3)
)

print(summary_df)

# Part 2 — Knowledge-Base Retrieval Test

This test evaluates the ability of each candidate embedding model to retrieve
clinically relevant evidence from the medical knowledge base.

Candidate retrieval methods:
- BAAI/bge-base-en-v1.5
- ncbi/MedCPT
- nomic-ai/nomic-embed-text-v1.5
- BM25 baseline

In [ ]:
# Load the four KB chunk files
KB_FOLDER = "Knowledge Base/Processed_v2"

KB_FILES = [
    "cardiology_chunks.jsonl",
    "infectious_diseases_chunks.jsonl",
    "nephrology_chunks.jsonl",
    "pulmonology_chunks.jsonl",
]

kb_chunks = []

for filename in KB_FILES:
    path = f"{KB_FOLDER}/{filename}"

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                kb_chunks.append(json.loads(line))

print(f"Total KB chunks loaded: {len(kb_chunks)}")

In [ ]:
# Inspect the structure
print("Example KB chunk:")
print(kb_chunks[0])

print("\nAvailable fields:")
print(kb_chunks[0].keys())

In [ ]:
# check how many chunks came from each specialty
from collections import Counter
specialty_counts = Counter(
    chunk["specialty"]
    for chunk in kb_chunks
)

for specialty, count in specialty_counts.items():
    print(f"{specialty}: {count} chunks")

In [ ]:
#Create specialty-specific KB corpora
kb_chunks_by_specialty = defaultdict(list)

for chunk in kb_chunks:
    kb_chunks_by_specialty[chunk["specialty"]].append(chunk)

for specialty, chunks in kb_chunks_by_specialty.items():
    print(f"{specialty}: {len(chunks)} chunks")

In [ ]:
# Create a simple KB inspection/search function
def search_kb_chunks(specialty, keywords, max_results=20):
    """
    Find KB chunks within one specialty containing the supplied keywords.
    Used only to help manually construct retrieval-test ground truth.
    """

    chunks = kb_chunks_by_specialty[specialty]

    keywords = [keyword.lower() for keyword in keywords]

    matches = []

    for chunk in chunks:
        text = chunk["text"].lower()

        if any(keyword in text for keyword in keywords):
            matches.append(chunk)

    print(f"Found {len(matches)} matching chunks.\n")

    for chunk in matches[:max_results]:
        print(f"Chunk ID: {chunk['chunk_id']}")
        print(f"Source: {chunk['source_file']}")
        print(f"Headings: {chunk['headings']}")
        print(f"Text: {chunk['text'][:700]}")
        print("-" * 80)

    return matches

In [ ]:
matches = search_kb_chunks(
    specialty="Cardiology",
    keywords=["heart failure"],
    max_results=20
)

In [ ]:
matches = search_kb_chunks(
    specialty="Cardiology",
    keywords=[
        "guideline-directed medical therapy",
        "guideline-directed management",
        "systolic dysfunction"
    ],
    max_results=20
)

In [ ]:
# Cardiology query 1
KB_RETRIEVAL_TESTS = [
    {
        "specialty": "Cardiology",
        "query": "What guideline-directed medical therapy is recommended for patients with chronic severe secondary mitral regurgitation and heart failure with reduced ejection fraction?",
        "relevant_chunk_ids": [
            "cardiology_chunk_000230"
        ]
    }
]

In [ ]:
matches = search_kb_chunks(
    specialty="Cardiology",
    keywords=[
        "cardiac dysrhythmia",
        "cardiac arrhythmia",
        "arrhythmia",
        "atrial fibrillation"
    ],
    max_results=20
)

In [ ]:
# Cardiology query 2
KB_RETRIEVAL_TESTS.append({
    "specialty": "Cardiology",
    "query": (
        "What oral anticoagulation is recommended for patients "
        "with valvular heart disease and atrial fibrillation, "
        "including those with rheumatic mitral stenosis or a mechanical prosthesis?"
    ),
    "relevant_chunk_ids": [
        "cardiology_chunk_000003"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Cardiology",
    keywords=[
        "secondary hypertension",
        "resistant hypertension",
        "hypertensive emergency",
        "hypertensive crisis",
        "target organ damage"
    ],
    max_results=20
)

In [ ]:
# Cardiology query 3
KB_RETRIEVAL_TESTS.append({
    "specialty": "Cardiology",
    "query": (
        "What clinical features should raise suspicion "
        "for secondary hypertension?"
    ),
    "relevant_chunk_ids": [
        "cardiology_chunk_000825"
    ]
})

In [ ]:
# matches = search_kb_chunks(
#     specialty="Cardiology",
#     keywords=[
#         "coronary atherosclerosis",
#         "coronary artery disease",
#         "chronic coronary disease",
#         "myocardial ischemia",
#         "atherosclerotic cardiovascular disease"
#     ],
#     max_results=20
# )

In [ ]:
# # Cardiology query 4
# KB_RETRIEVAL_TESTS.append({
#     "specialty": "Cardiology",
#     "query": (
#         "How should coronary artery disease risk factors "
#         "be managed after valve intervention?"
#     ),
#     "relevant_chunk_ids": [
#         "cardiology_chunk_000069"
#     ]
# })

In [ ]:
# One small concern, though: this chunk is CAD in the context of valvular disease, rather than coronary atherosclerosis generally. 
# That's acceptable if your disease category is broad, but if your KB contains a dedicated chronic coronary disease/atherosclerosis guideline, 
# I would prefer a chunk from that source for this test because it would represent your fourth condition more directly.
# So before locking query 4, I recommend checking whether your cardiology KB has a dedicated coronary-disease source. We can do that with a tiny code cell next.
    
coronary_sources = sorted({
    chunk["source_file"]
    for chunk in kb_chunks_by_specialty["Cardiology"]
    if any(
        term in chunk["text"].lower()
        for term in [
            "coronary artery disease",
            "coronary atherosclerosis",
            "chronic coronary disease"
        ]
    )
})

for source in coronary_sources:
    print(source)

In [ ]:
# The above is much more appropriate for condition 4 than the valvular-disease chunk we previously selected.
# Let's search only that source for a focused recommendation:

ccd_matches = [
    chunk
    for chunk in kb_chunks_by_specialty["Cardiology"]
    if "CIR011" in chunk["source_file"]
    and any(
        term in chunk["text"].lower()
        for term in [
            "antiplatelet therapy",
            "aspirin",
            "statin",
            "lipid-lowering",
            "revascularization"
        ]
    )
]

print(f"Found {len(ccd_matches)} matching chunks.\n")

for chunk in ccd_matches[:20]:
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Headings: {chunk['headings']}")
    print(f"Text: {chunk['text'][:700]}")
    print("-" * 80)

In [ ]:
# Cardiology query 4
KB_RETRIEVAL_TESTS.append({
    "specialty": "Cardiology",
    "query": (
        "How should patients with chronic coronary disease "
        "and persistent stable chest pain be evaluated and managed?"
    ),
    "relevant_chunk_ids": [
        "cardiology_chunk_001706"
    ]
})

In [ ]:
for i, test in enumerate(KB_RETRIEVAL_TESTS, start=1):
    print(i, test["specialty"], test["query"])

In [ ]:
matches = search_kb_chunks(
    specialty="Cardiology",
    keywords=[
        "primary hypertension",
        "essential hypertension",
        "first-line antihypertensive",
        "first-line agents",
        "initial antihypertensive therapy",
        "blood pressure treatment"
    ],
    max_results=20
)

In [ ]:
# Cardiology query 5
KB_RETRIEVAL_TESTS.append({
    "specialty": "Cardiology",
    "query": (
        "What medication classes are recommended as "
        "first-line pharmacological therapy for primary hypertension?"
    ),
    "relevant_chunk_ids": [
        "cardiology_chunk_000886"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Infectious Diseases",
    keywords=[
        "bacterial infection",
        "bacterial infections",
        "empiric antibiotic",
        "empiric antimicrobial",
        "antibiotic therapy",
        "antimicrobial therapy",
        "culture and susceptibility"
    ],
    max_results=20
)

In [ ]:
# Infectious Diseases query 1
KB_RETRIEVAL_TESTS.append({
    "specialty": "Infectious Diseases",
    "query": (
        "When should blood cultures be collected in adults "
        "with suspected sepsis in relation to antimicrobial therapy?"
    ),
    "relevant_chunk_ids": [
        "infectious_diseases_chunk_000051"
    ]
})

In [ ]:
# Infectious Diseases query 2
KB_RETRIEVAL_TESTS.append({
    "specialty": "Infectious Diseases",
    "query": (
        "When should antimicrobial therapy be initiated "
        "in adults with sepsis or septic shock?"
    ),
    "relevant_chunk_ids": [
        "infectious_diseases_chunk_000071"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Infectious Diseases",
    keywords=[
        "community-acquired pneumonia",
        "hospital-acquired pneumonia",
        "bacterial pneumonia",
        "pneumonia antibiotic",
        "pneumonia antimicrobial",
        "empiric treatment pneumonia"
    ],
    max_results=20
)

In [ ]:
# Infectious Diseases query 3
KB_RETRIEVAL_TESTS.append({
    "specialty": "Infectious Diseases",
    "query": (
        "What principles should guide the initiation and selection "
        "of empiric antibiotic therapy for community-acquired bacterial pneumonia?"
    ),
    "relevant_chunk_ids": [
        "infectious_diseases_chunk_000993"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Infectious Diseases",
    keywords=[
        "urinary tract infection",
        "complicated urinary tract infection",
        "pyelonephritis",
        "cystitis",
        "urine culture",
        "empiric therapy urinary"
    ],
    max_results=20
)

In [ ]:
# Infectious Diseases query 4
KB_RETRIEVAL_TESTS.append({
    "specialty": "Infectious Diseases",
    "query": (
        "What factors should be considered when selecting empiric therapy "
        "for an uncomplicated urinary tract infection when ESBL-producing "
        "organisms may be present?"
    ),
    "relevant_chunk_ids": [
        "infectious_diseases_chunk_000382"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Infectious Diseases",
    keywords=[
        "invasive candidiasis",
        "candidemia",
        "empiric antifungal",
        "initial therapy candidemia",
        "echinocandin",
        "invasive fungal infection"
    ],
    max_results=20
)


In [ ]:
# Infectious Diseases query 5
KB_RETRIEVAL_TESTS.append({
    "specialty": "Infectious Diseases",
    "query": (
        "What is the recommended initial antifungal therapy "
        "for candidemia in nonneutropenic adult patients?"
    ),
    "relevant_chunk_ids": [
        "infectious_diseases_chunk_000646"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "acute kidney injury",
        "acute renal failure",
        "AKI",
        "serum creatinine",
        "urine output",
        "renal replacement therapy"
    ],
    max_results=20
)

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "AKI is defined",
        "acute kidney injury is defined",
        "increase in serum creatinine",
        "0.3 mg/dl",
        "urine volume",
        "stage 1 AKI",
        "stage 2 AKI",
        "stage 3 AKI"
    ],
    max_results=20
)

In [ ]:
# Nephrology query 1
KB_RETRIEVAL_TESTS.append({
    "specialty": "Nephrology",
    "query": (
        "What clinical criteria are used to define acute kidney injury "
        "based on changes in serum creatinine and urine output?"
    ),
    "relevant_chunk_ids": [
        "nephrology_chunk_000881"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "fluid overload",
        "volume overload",
        "hyperkalemia",
        "hypokalemia",
        "hyponatremia",
        "hypernatremia",
        "electrolyte disorder",
        "electrolyte abnormalities"
    ],
    max_results=20
)

In [ ]:
# Nephrology query 2
KB_RETRIEVAL_TESTS.append({
    "specialty": "Nephrology",
    "query": (
        "How should fluid and electrolyte disturbances such as edema, "
        "abnormal potassium levels, and hyponatremia be managed "
        "in patients with nephrotic syndrome?"
    ),
    "relevant_chunk_ids": [
        "nephrology_chunk_000294"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "chronic kidney disease",
        "CKD staging",
        "CKD progression",
        "eGFR less than 60",
        "albuminuria",
        "kidney failure risk",
        "management of CKD"
    ],
    max_results=20
)

In [ ]:
# Nephrology query 3
KB_RETRIEVAL_TESTS.append({
    "specialty": "Nephrology",
    "query": (
        "How is chronic kidney disease defined and classified "
        "according to KDIGO?"
    ),
    "relevant_chunk_ids": [
        "nephrology_chunk_000061"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "kidney disease",
        "renal disease",
        "tubulointerstitial",
        "interstitial nephritis",
        "obstructive nephropathy",
        "urinary obstruction",
        "hydronephrosis",
        "ureter"
    ],
    max_results=20
)

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "acute interstitial nephritis",
        "chronic interstitial nephritis",
        "tubulointerstitial nephritis",
        "obstructive uropathy",
        "obstructive nephropathy",
        "urinary tract obstruction",
        "hydronephrosis"
    ],
    max_results=20
)

In [ ]:
# Nephrology query 4
KB_RETRIEVAL_TESTS.append({
    "specialty": "Nephrology",
    "query": (
        "How should urinary tract obstruction be evaluated "
        "in a patient with suspected obstructive kidney injury?"
    ),
    "relevant_chunk_ids": [
        "nephrology_chunk_001056"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Nephrology",
    keywords=[
        "glomerulonephritis",
        "nephritic syndrome",
        "nephrotic syndrome",
        "glomerular disease",
        "proteinuria hematuria",
        "kidney biopsy",
        "renal sclerosis",
        "glomerulosclerosis"
    ],
    max_results=20
)

In [ ]:
# Nephrology query 5
KB_RETRIEVAL_TESTS.append({
    "specialty": "Nephrology",
    "query": (
        "What treatment options are considered for adults with "
        "steroid-resistant focal segmental glomerulosclerosis "
        "and nephrotic syndrome?"
    ),
    "relevant_chunk_ids": [
        "nephrology_chunk_000039"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Pulmonology",
    keywords=[
        "acute respiratory failure",
        "hypoxemic respiratory failure",
        "hypercapnic respiratory failure",
        "respiratory insufficiency",
        "mechanical ventilation",
        "noninvasive ventilation",
        "oxygen therapy",
        "respiratory arrest"
    ],
    max_results=20
)

In [ ]:
#Pulmonology query 1
KB_RETRIEVAL_TESTS.append({
    "specialty": "Pulmonology",
    "query": (
        "How should oxygen therapy be managed in hospitalized patients, "
        "particularly those at risk of hypercapnic respiratory failure?"
    ),
    "relevant_chunk_ids": [
        "pulmonology_chunk_000032"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Pulmonology",
    keywords=[
        "community acquired pneumonia",
        "community-acquired pneumonia",
        "pneumonia antibiotic",
        "empirical antibiotic",
        "empiric antibiotic",
        "pneumonia severity",
        "CURB65",
        "severe pneumonia"
    ],
    max_results=20
)


In [ ]:
#Pulmonology query 2
KB_RETRIEVAL_TESTS.append({
    "specialty": "Pulmonology",
    "query": (
        "How should the severity of community-acquired pneumonia "
        "be assessed in hospitalized adults using the CURB65 score?"
    ),
    "relevant_chunk_ids": [
        "pulmonology_chunk_000028"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Pulmonology",
    keywords=[
        "COPD exacerbation",
        "acute exacerbation of COPD",
        "chronic obstructive pulmonary disease",
        "bronchodilator COPD",
        "noninvasive ventilation COPD",
        "bronchiectasis exacerbation",
        "bronchiectasis antibiotics",
        "bronchiectasis management"
    ],
    max_results=20
)

In [ ]:
# Pulmonology query 3
KB_RETRIEVAL_TESTS.append({
    "specialty": "Pulmonology",
    "query": (
        "Which hospitalized patients with an acute COPD exacerbation "
        "should receive antibiotic therapy?"
    ),
    "relevant_chunk_ids": [
        "pulmonology_chunk_001400"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Pulmonology",
    keywords=[
        "pleural effusion",
        "pleural fluid",
        "thoracentesis",
        "pleural aspiration",
        "pleurisy",
        "pleuritic pain",
        "pulmonary collapse",
        "lung collapse",
        "atelectasis"
    ],
    max_results=20
)

In [ ]:
# Pulmonology query 4
KB_RETRIEVAL_TESTS.append({
    "specialty": "Pulmonology",
    "query": (
        "How should a parapneumonic pleural effusion be managed, "
        "and when is pleural fluid drainage indicated?"
    ),
    "relevant_chunk_ids": [
        "pulmonology_chunk_000053"
    ]
})

In [ ]:
matches = search_kb_chunks(
    specialty="Pulmonology",
    keywords=[
        "aspiration pneumonitis",
        "chemical pneumonitis",
        "aspiration pneumonia",
        "aspiration event",
        "gastric aspiration",
        "antibiotics aspiration pneumonitis",
        "macroaspiration"
    ],
    max_results=20
)

In [ ]:
# Pulmonology query 5
KB_RETRIEVAL_TESTS.append({
    "specialty": "Pulmonology",
    "query": (
        "How can aspiration pneumonitis be distinguished from aspiration pneumonia, "
        "and when are antibiotics indicated?"
    ),
    "relevant_chunk_ids": [
        "pulmonology_chunk_000906"
    ]
})

In [ ]:
# ============================================================
# Remove the duplicate CIR033 guideline (same PDF present twice)
# ============================================================
import json, csv, shutil
from pathlib import Path

CARD = Path("Knowledge Base/Processed_v2/cardiology_chunks.jsonl")
rows = [json.loads(l) for l in open(CARD, encoding="utf-8") if l.strip()]

copies = sorted({r["source_file"] for r in rows if "blooda-31203-main" in r["source_file"]})
print("Copies found:")
for s in copies:
    print(f"  {s!r}: {sum(r['source_file'] == s for r in rows)} chunks")

# Which copy do the smoke-test labels use?
labelled = {cid for t in KB_RETRIEVAL_TESTS for cid in t["relevant_chunk_ids"]}
used = {s: sorted(r["chunk_id"] for r in rows if r["source_file"] == s and r["chunk_id"] in labelled)
        for s in copies}
print("Labelled chunks per copy:", used)

if len(copies) != 2:
    print("\nSTOP: expected exactly 2 copies - nothing changed.")
elif all(used[s] for s in copies):
    print("\nSTOP: both copies have labelled chunks - nothing changed. Send me this output.")
else:
    drop = next((s for s in copies if not used[s]), copies[1])
    keep_rows = [r for r in rows if r["source_file"] != drop]
    dropped   = [r for r in rows if r["source_file"] == drop]

    with open(CARD, "w", encoding="utf-8") as f:
        for r in keep_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    with open(CARD.parent / "removed_duplicate_chunks.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["chunk_id", "source_file"])
        w.writerows([(r["chunk_id"], r["source_file"]) for r in dropped])

    # Move the duplicate PDF out of the Cardiology folder
    pdf = Path(dropped[0]["source_path"])
    dest = Path("Knowledge Base/_removed_duplicates")
    dest.mkdir(exist_ok=True)
    if pdf.exists():
        shutil.move(str(pdf), str(dest / pdf.name))

    print(f"\nDropped copy: {drop!r} ({len(dropped)} chunks)")
    print(f"Cardiology chunks now: {len(keep_rows):,}")
    print("Unique cardiology guidelines now:", len({r['source_file'] for r in keep_rows}))

In [ ]:
import json
from collections import Counter
from pathlib import Path

for fn in ["cardiology", "infectious_diseases", "nephrology", "pulmonology"]:
    rows = [json.loads(l) for l in open(Path("Knowledge Base/Processed_v2") / f"{fn}_chunks.jsonl", encoding="utf-8") if l.strip()]
    srcs = Counter(r["source_file"] for r in rows)
    print(f"\n{fn}: {len(srcs)} guidelines, {len(rows):,} chunks")
    for s, n in sorted(srcs.items()):
        print(f"  {n:>5}  {s}")

In [ ]:
# ============================================================
# Check: do all KB relevant chunks still exist after reference removal?
# ============================================================
kb_ids = {c["chunk_id"] for c in kb_chunks}
print(f"KB chunks loaded: {len(kb_chunks):,}  (expected 11,576)\n")

missing_any = False
for i, test in enumerate(KB_RETRIEVAL_TESTS, start=1):
    gone = [cid for cid in test["relevant_chunk_ids"] if cid not in kb_ids]
    status = "OK" if not gone else f"REMOVED: {gone}"
    missing_any = missing_any or bool(gone)
    print(f"{i:>2} | {test['specialty']:20s} | {len(test['relevant_chunk_ids'])} relevant | {status}")

print("\nAll KB labels valid." if not missing_any else "\nSome labelled chunks were removed - send me this output.")

In [ ]:
# Knowledge-Base Retrieval Evaluation
# verify that the KB chunk IDs are unique within each specialty
print("Checking KB chunk-ID uniqueness:\n")

for specialty, chunks in kb_chunks_by_specialty.items():
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    print(
        f"{specialty}: "
        f"{len(chunk_ids)} chunks, "
        f"{len(set(chunk_ids))} unique IDs"
    )

In [ ]:
# Building the KB embedding cache.
kb_embedding_cache = {}



In [ ]:
# Building the KB embedding cache for Cardiology
kb_embedding_cache = {}

specialty = "Cardiology"
model_name = "bge-base-en-v1.5"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

chunk_ids = [
    chunk["chunk_id"]
    for chunk in chunks
]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
def embed_with_nomic(texts, is_query=False, batch_size=32):

    prefix = "search_query: " if is_query else "search_document: "

    prefixed_texts = [
        prefix + text
        for text in texts
    ]

    embeddings = nomic_model.encode(
        prefixed_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [ ]:
import inspect

print(inspect.signature(embed_with_nomic))
print(inspect.signature(MODEL_REGISTRY["nomic-embed-text"]))

In [ ]:
MODEL_REGISTRY["nomic-embed-text"] = embed_with_nomic

In [ ]:
print(inspect.signature(embed_with_nomic))
print(inspect.signature(MODEL_REGISTRY["nomic-embed-text"]))

In [ ]:
specialty = "Cardiology"
model_name = "nomic-embed-text"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
specialty = "Cardiology"
model_name = "medcpt"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
import os
import numpy as np

os.makedirs("Knowledge Base/Embeddings", exist_ok=True)

specialty = "Cardiology"

for model_name in EMBEDDING_MODELS:

    cached = kb_embedding_cache[(model_name, specialty)]

    filename = (
        f"Knowledge Base/Embeddings/"
        f"cardiology_{model_name}.npz"
    )

    np.savez_compressed(
        filename,
        chunk_ids=np.array(cached["chunk_ids"]),
        embeddings=cached["embeddings"]
    )

    print(f"Saved: {filename}")

print("\nAll Cardiology embeddings saved successfully.")

In [ ]:
# Infectious Diseases — BGE

specialty = "Infectious Diseases"
model_name = "bge-base-en-v1.5"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Infectious Diseases — Nomic

specialty = "Infectious Diseases"
model_name = "nomic-embed-text"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# This step is needed after updating embed_with_medcpt in the cell4 above to add the progress bar.
MODEL_REGISTRY["medcpt"] = embed_with_medcpt

In [ ]:
# Infectious Diseases — MedCPT

specialty = "Infectious Diseases"
model_name = "medcpt"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Save Infectious Diseases embeddings

import os
import numpy as np

os.makedirs("Knowledge Base/Embeddings", exist_ok=True)

specialty = "Infectious Diseases"

for model_name in EMBEDDING_MODELS:

    cached = kb_embedding_cache[(model_name, specialty)]

    filename = (
        f"Knowledge Base/Embeddings/"
        f"infectious_diseases_{model_name}.npz"
    )

    np.savez_compressed(
        filename,
        chunk_ids=np.array(cached["chunk_ids"]),
        embeddings=cached["embeddings"]
    )

    print(f"Saved: {filename}")

print("\nAll Infectious Diseases embeddings saved successfully.")

In [ ]:
# Nephrology — BGE

specialty = "Nephrology"
model_name = "bge-base-en-v1.5"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Nephrology — Nomic

specialty = "Nephrology"
model_name = "nomic-embed-text"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Nephrology — MedCPT

specialty = "Nephrology"
model_name = "medcpt"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Save Nephrology embeddings

import os
import numpy as np

os.makedirs("Knowledge Base/Embeddings", exist_ok=True)

specialty = "Nephrology"

for model_name in EMBEDDING_MODELS:

    cached = kb_embedding_cache[(model_name, specialty)]

    filename = (
        f"Knowledge Base/Embeddings/"
        f"nephrology_{model_name}.npz"
    )

    np.savez_compressed(
        filename,
        chunk_ids=np.array(cached["chunk_ids"]),
        embeddings=cached["embeddings"]
    )

    print(f"Saved: {filename}")

print("\nAll Nephrology embeddings saved successfully.")

In [ ]:
# Pulmonology — BGE

specialty = "Pulmonology"
model_name = "bge-base-en-v1.5"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Pulmonology — Nomic

specialty = "Pulmonology"
model_name = "nomic-embed-text"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Pulmonology — MedCPT

specialty = "Pulmonology"
model_name = "medcpt"

chunks = kb_chunks_by_specialty[specialty]

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print(
    f"Embedding {len(chunks)} {specialty} chunks "
    f"with {model_name}..."
)

embeddings = MODEL_REGISTRY[model_name](
    chunk_texts,
    is_query=False,
    batch_size=32
)

kb_embedding_cache[(model_name, specialty)] = {
    "chunk_ids": chunk_ids,
    "embeddings": embeddings
}

print("Finished.")
print("Embedding shape:", embeddings.shape)

In [ ]:
# Save Pulmonology embeddings

import os
import numpy as np

os.makedirs("Knowledge Base/Embeddings", exist_ok=True)

specialty = "Pulmonology"

for model_name in EMBEDDING_MODELS:

    cached = kb_embedding_cache[(model_name, specialty)]

    filename = (
        f"Knowledge Base/Embeddings/"
        f"pulmonology_{model_name}.npz"
    )

    np.savez_compressed(
        filename,
        chunk_ids=np.array(cached["chunk_ids"]),
        embeddings=cached["embeddings"]
    )

    print(f"Saved: {filename}")

print("\nAll Pulmonology embeddings saved successfully.")

In [ ]:
# Cached KB retrieval function
# For each of the 20 queries, it will:
# query → embed query → compare against saved/cached specialty embeddings → rank chunks → return top 10

def retrieve_kb_with_cached_embeddings(
    model_name,
    specialty,
    query,
    top_k=10
):
    """
    Retrieve the most relevant KB chunks for a query
    using the precomputed embeddings for the corresponding specialty.
    """

    embed_function = MODEL_REGISTRY[model_name]

    cached = kb_embedding_cache[(model_name, specialty)]

    chunk_ids = cached["chunk_ids"]
    chunk_embeddings = cached["embeddings"]

    # Embed only the query
    query_embedding = embed_function(
        [query],
        is_query=True
    )[0]

    # Cosine similarity because embeddings are normalized
    scores = chunk_embeddings @ query_embedding

    ranked_indices = np.argsort(scores)[::-1]

    ranked_ids = [
        chunk_ids[i]
        for i in ranked_indices[:top_k]
    ]

    return ranked_ids

In [ ]:
# The BM25 retrieval function
def retrieve_kb_with_bm25(
    specialty,
    query,
    top_k=10
):
    """
    Retrieve KB chunks using BM25 lexical retrieval
    within the corresponding specialty.
    """

    from rank_bm25 import BM25Okapi

    chunks = kb_chunks_by_specialty[specialty]

    chunk_texts = [chunk["text"] for chunk in chunks]
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    tokenized_corpus = [
        text.lower().split()
        for text in chunk_texts
    ]

    tokenized_query = query.lower().split()

    bm25 = BM25Okapi(tokenized_corpus)

    scores = bm25.get_scores(tokenized_query)

    ranked_indices = np.argsort(scores)[::-1]

    ranked_ids = [
        chunk_ids[i]
        for i in ranked_indices[:top_k]
    ]

    return ranked_ids

In [ ]:
# Evaluate KB retrieval
# Run the actual KB retrieval evaluation across all 20 queries and all four retrieval methods.
KB_TOP_K = 10

kb_results = []

for test_number, test in enumerate(KB_RETRIEVAL_TESTS, start=1):

    specialty = test["specialty"]
    query = test["query"]
    relevant_ids = test["relevant_chunk_ids"]

    print(
        f"Test {test_number}/{len(KB_RETRIEVAL_TESTS)} "
        f"- {specialty}"
    )

    # Dense embedding models
    for model_name in EMBEDDING_MODELS:

        ranked_ids = retrieve_kb_with_cached_embeddings(
            model_name=model_name,
            specialty=specialty,
            query=query,
            top_k=KB_TOP_K
        )

        kb_results.append({
            "test": test_number,
            "specialty": specialty,
            "model": model_name,
            "recall@1": recall_at_k(
                ranked_ids, relevant_ids, 1
            ),
            "recall@3": recall_at_k(
                ranked_ids, relevant_ids, 3
            ),
            "recall@5": recall_at_k(
                ranked_ids, relevant_ids, 5
            ),
            "recall@10": recall_at_k(
                ranked_ids, relevant_ids, 10
            ),
            "mrr@10": mrr_at_k(
                ranked_ids, relevant_ids, 10
            )
        })

    # BM25 baseline
    ranked_ids = retrieve_kb_with_bm25(
        specialty=specialty,
        query=query,
        top_k=KB_TOP_K
    )

    kb_results.append({
        "test": test_number,
        "specialty": specialty,
        "model": "BM25",
        "recall@1": recall_at_k(
            ranked_ids, relevant_ids, 1
        ),
        "recall@3": recall_at_k(
            ranked_ids, relevant_ids, 3
        ),
        "recall@5": recall_at_k(
            ranked_ids, relevant_ids, 5
        ),
        "recall@10": recall_at_k(
            ranked_ids, relevant_ids, 10
        ),
        "mrr@10": mrr_at_k(
            ranked_ids, relevant_ids, 10
        )
    })

print(
    f"\nFinished. Total evaluations: "
    f"{len(kb_results)}"
)

In [ ]:
import pandas as pd

kb_results_df = pd.DataFrame(kb_results)

kb_summary_df = (
    kb_results_df
    .groupby("model")[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr@10"]
    ]
    .mean()
    .round(3)
)

print(kb_summary_df)

In [ ]:
specialty_summary_df = (
    kb_results_df
    .groupby(["specialty", "model"])[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr@10"]
    ]
    .mean()
    .round(3)
)

print(specialty_summary_df)

In [ ]:
import numpy as np

def paired_perm_p(a, b, n=10000, seed=0):
    rng = np.random.default_rng(seed)
    d = np.asarray(a) - np.asarray(b)
    obs = abs(d.mean())
    flips = rng.choice([-1, 1], size=(n, len(d)))
    return ((np.abs((flips * d).mean(axis=1)) >= obs).sum() + 1) / (n + 1)

for df, name in [(results_df, "Patient"), (kb_results_df, "KB")]:
    piv = df.pivot_table(index="test", columns="model", values="mrr@10")
    for other in [m for m in piv.columns if m != "bge-base-en-v1.5"]:
        diff = (piv["bge-base-en-v1.5"] - piv[other]).mean()
        p = paired_perm_p(piv["bge-base-en-v1.5"], piv[other])
        print(f"{name}: BGE vs {other:18s} diff={diff:+.3f}  p={p:.3f}")